# 🛡️ End-to-End AI Credit Risk Prediction & SHAP Explainability

This notebook walks through the complete credit risk underwriting pipeline:
1. **Data Acquisition & SQLite Database Ingestion** (`Give Me Some Credit` schema)
2. **SQL Analytics** (Default rate by segment, `NTILE(10)` window functions, and multi-table `JOIN`s)
3. **Data Cleaning, EDA & Domain Feature Engineering** (`debt_to_income`, `credit_utilization`, `payment_to_income`, `age_bucket`, `delinquency_flags`)
4. **Stratified Split & Class Imbalance Handling** (`SMOTE` & class weights)
5. **Model Training & Optuna Hyperparameter Tuning** (Logistic Regression, Random Forest, XGBoost, LightGBM)
6. **Evaluation Metrics** (ROC-AUC, Gini, KS Statistic, Precision, Recall, F1) & Configurable **Low / Medium / High** Risk Tiers
7. **SHAP Global & Per-Customer Explainability**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import joblib
import pandas as pd
from src.config import BEST_MODEL_PATH, SQLITE_DB_PATH
from src.database import run_sql_analysis, load_applicants_from_db
from src.explainability import explain_single_applicant, plot_customer_shap_contributions
from src.model_training import predict_credit_risk

## 1. SQLite Analytics: Segment Default Rates, Window Functions & Multi-Table Joins

In [ ]:
sql_results = run_sql_analysis(SQLITE_DB_PATH)
for query_name, df_res in sql_results.items():
    print(f'=== SQL Query: {query_name} ===')
    display(df_res.head(10))

## 2. Model Benchmark Comparison (ROC-AUC, Gini, KS, Precision, Recall, F1)

In [ ]:
bundle = joblib.load(BEST_MODEL_PATH)
metrics_df = pd.DataFrame(bundle['metrics_table'])
display(metrics_df)

## 3. Global & Per-Customer SHAP Explanations

In [ ]:
shap_global_df = pd.DataFrame(bundle['shap_global_importance'])
display(shap_global_df.head(10))

# Score and explain a single sample borrower
df_sample = load_applicants_from_db(SQLITE_DB_PATH).iloc[[12]]
scored = predict_credit_risk(df_sample, bundle)
reasons = explain_single_applicant(scored, bundle, top_k=6)
print(f"Predicted PD: {scored['predicted_pd'].iloc[0]:.2%} | Tier: {scored['risk_tier'].iloc[0]} | Credit Score: {scored['credit_score'].iloc[0]}")
fig = plot_customer_shap_contributions(reasons, title_suffix='(Applicant #13)')